# Defiende el servidor — Versionado de código, datos, experimentos y modelos
**MLOps · 2026-II · Taller evaluable** · Clasificación binaria normal/ataque con UNSW-NB15

Este cuaderno recorre **todo el taller de principio a fin**, con gráficas y resultados en cada paso:

| # | Sección | Herramienta | Evidencia que produce |
|---|---|---|---|
| 1 | Configuración | — | parámetros del proyecto |
| 2 | Datos originales | DVC + Git | `data/raw.dvc`, huellas SHA-256 |
| 3 | Auditoría y particiones sin fuga | pandas | `evidencias/auditoria.json`, gráficas |
| 4 | Datos preparados | DVC + Git | `data/prepared.dvc` |
| 5 | Experimento E1 y versión **v1** | MLflow + Git | run ID, `hf_v1/`, tag `v1` |
| 6 | Experimentos E2–E4 y comparación | MLflow | `evidencias/experimentos.csv` |
| 7 | Selección de umbral y versión **v2** | validación | `hf_v2/`, tag `v2` |
| 8 | Publicación en Hugging Face | HF Hub | SHA de v1 y v2 |
| 9 | Congelar la selección | Git | `evidencias/seleccion.json`, tag |
| 10 | Prueba de recuperación | — | `evidencias/recuperacion/` |
| 11 | Evaluación final en test (**una sola vez**) | — | `metricas-test.json`, `predicciones-test.csv` |
| 12 | Exportación MLflow e índice de entrega | MLflow | `evidencias/mlflow/`, `entrega.md` |

> **Cadena de trazabilidad:** commit Git → referencia DVC → run ID de MLflow → revisión (SHA) de Hugging Face.

**Cómo usarlo.** Ejecuta las celdas **en orden** desde la raíz del proyecto (VS Code: abre la carpeta `proyecto/`; Colab: clona/sube la carpeta y `%cd` a ella).
Las celdas que tocan servicios externos (GitHub, DVC remoto, Hugging Face) están protegidas por banderas en la sección 1 y **necesitan tus credenciales** (nunca las pegues en el cuaderno).

## 1. Configuración
Parámetros únicos del proyecto. Cambia solo lo marcado con `← EDITA`.

In [1]:
import os, sys, json, shutil, subprocess, warnings, datetime as dt
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yaml, joblib, mlflow
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_curve, precision_recall_curve, ConfusionMatrixDisplay
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.titlesize": 11})

ROOT = Path.cwd()                      # el cuaderno se ejecuta desde la raíz del proyecto
sys.path.insert(0, str(ROOT))
from src.common import (SEED, TARGET, EXCLUDE, sha256_file, sha256_array, git_commit, git_dirty, dvc_reference,
                        data_version_string, feature_columns, row_group_hash, compute_metrics, apply_threshold,
                        pick_threshold, predict_with_bundle)
from src.run_experiment import run_experiment, load_configs, setup_mlflow, load_prepared, EXPERIMENT_NAME
from src.bundle import build_bundle, publish_version

# ---- Parámetros del proyecto ------------------------------------------------------------------
MAX_FPR   = 0.05          # criterio de umbral: maximizar recall de ataque con FPR <= 5 % (se decide SOLO con validación)
N_FOLDS   = 5             # validación = 1 de 5 pliegues agrupados (~20 %)
TRAIN_FILE, TEST_FILE = "UNSW_NB15_training-set.csv", "UNSW_NB15_testing-set.csv"   # solo el nombre; los CSV viven en data/raw/
EXPECTED_SHA = {TRAIN_FILE: "bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa",
                TEST_FILE:  "734fe6642edf758f7c94d7d9149426b49d202fe8e7bf0bef47392489c3c0a559"}   # anexo del taller (mirror)
EXPECTED_ROWS = {TRAIN_FILE: 175_341, TEST_FILE: 82_332}
SOURCE_URLS = {TRAIN_FILE: "", TEST_FILE: ""}       # ← EDITA: pega aquí los enlaces del anexo, o copia los CSV a data/raw/
SOURCE_INFO = "UNSW-NB15 (UNSW Canberra). Copia fija (mirror de terceros) enlazada en el taller; atribución: página oficial de UNSW."
DOWNLOAD_DATE = ""                                   # ← EDITA: 'AAAA-MM-DD'; si queda vacío se usa la fecha de modificación del archivo

HF_REPO_ID  = os.environ.get("LemonBynon", "LemonBynon/defiende-el-servidor-unsw")   # ← EDITA (o variable de entorno)
HF_PRIVATE  = False        # público es lo más simple; privado con acceso para el docente exige organización en HF
PUBLISH_HF  = False     # ← pon True cuando tengas HF_TOKEN en el entorno (sección 8)
DVC_REMOTE_URL = os.environ.get("DVC_REMOTE_URL", "https://dagshub.com/davidprogramacion04-arch/data_taller_versiones.dvc")  # ← EDITA: remote accesible al docente (DagsHub, S3, Drive, SSH…). Nunca solo local.

RAW, PREP, EV, FIG = ROOT/"data/raw", ROOT/"data/prepared", ROOT/"evidencias", ROOT/"evidencias/figuras"
for d in (RAW, PREP, EV, FIG, EV/"recuperacion", EV/"mlflow"): d.mkdir(parents=True, exist_ok=True)
# Si pusiste los CSV directamente en data/, los mueve a data/raw/ (que es la carpeta que versiona DVC)
for _n in (TRAIN_FILE, TEST_FILE):
    if not (RAW/_n).exists() and (ROOT/"data"/_n).exists(): shutil.move(str(ROOT/"data"/_n), str(RAW/_n))

def sh(cmd, show=True):
    """Ejecuta un comando de shell desde la raíz y muestra la salida real (útil como evidencia)."""
    r = subprocess.run(cmd, shell=True, cwd=ROOT, capture_output=True, text=True)
    if show: print(f"$ {cmd}\n{(r.stdout + r.stderr).strip()}\n[salida: {r.returncode}]\n")
    return r

def savefig(name):
    plt.tight_layout(); plt.savefig(FIG/name, dpi=130, bbox_inches="tight"); plt.show()

print("Raíz del proyecto:", ROOT); print("Python:", sys.version.split()[0]); print("Commit actual:", git_commit(ROOT)[:12])

c:\Users\Usuario\anaconda3\envs\defiende-servidor\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raíz del proyecto: c:\Users\Usuario\Documents\taller_defiende_mlops
Python: 3.12.16
Commit actual: c007fc08146f


## 2. Datos originales (DVC) y huellas
Se usan **solo** las dos particiones publicadas (`training-set` y `testing-set`). El nombre del archivo no garantiza que sea la partición correcta,
así que se comprueban **tamaños y huellas SHA-256** contra el anexo del taller.

In [2]:
import urllib.request
for name, url in SOURCE_URLS.items():
    if not (RAW/name).exists() and url:
        print("Descargando", name); urllib.request.urlretrieve(url, RAW/name)
faltan = [n for n in (TRAIN_FILE, TEST_FILE) if not (RAW/n).exists()]
if faltan:
    raise FileNotFoundError(f"Copia {faltan} en {RAW} (o pega las URL en SOURCE_URLS) y vuelve a ejecutar.")
huellas = pd.DataFrame({n: {"sha256": sha256_file(RAW/n), "esperado": EXPECTED_SHA[n]} for n in EXPECTED_SHA}).T
huellas["coincide"] = huellas.sha256 == huellas.esperado
display(huellas)
if not huellas.coincide.all():
    print("⚠ ATENCIÓN: alguna huella no coincide con el anexo. Si usaste el sitio oficial puede ser normal; "
          "si usaste el mirror, descarga de nuevo. Queda registrado en auditoria.json.")

,sha256,esperado,coincide
C:\Users\Usuario\Documents\taller_defiende_mlops\data\UNSW_NB15_training-set.csv,bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675...,bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675...,True
C:\Users\Usuario\Documents\taller_defiende_mlops\data\UNSW_NB15_testing-set.csv,734fe6642edf758f7c94d7d9149426b49d202fe8e7bf0b...,734fe6642edf758f7c94d7d9149426b49d202fe8e7bf0b...,True


### 2.1 Primera versión de datos: archivos recibidos
Inicializa Git y DVC (si no existen) y versiona `data/raw` con DVC. Ejecuta esta celda **una vez**; puedes repetirla sin problema.

In [6]:
sh("git rev-parse --is-inside-work-tree || git init")
if not (ROOT/".dvc").exists(): sh("dvc init")
sh("dvc add data/raw")
sh("git add -A")
sh('git commit -m "Datos originales UNSW-NB15 versionados con DVC (version raw)"')
sh("git tag -f data-raw-v1")
print("Referencia DVC:", dvc_reference(ROOT))

$ git rev-parse --is-inside-work-tree || git init
true
[salida: 0]

$ dvc init
"dvc" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
[salida: 1]

$ dvc add data/raw
"dvc" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
[salida: 1]

$ git add -A

[salida: 0]

$ git commit -m "Datos originales UNSW-NB15 versionados con DVC (version raw)"
On branch master
nothing to commit, working tree clean
[salida: 1]

$ git tag -f data-raw-v1

[salida: 0]

Referencia DVC: {'raw': None, 'prepared': None}


## 3. Auditoría de datos y particiones sin fuga
**Qué se comprueba** (todo se guarda en `evidencias/auditoria.json`):
1. Fuente, SHA-256, filas, columnas, clases, faltantes y tipos de **cada archivo**.
2. Duplicados **por variables predictoras** (sin `id`, `label`, `attack_cat`) y grupos de entradas iguales con **etiquetas contradictorias**.
3. División entrenamiento/validación **solo** con el archivo de entrenamiento, por **grupos de entradas idénticas** (ningún grupo queda repartido).
4. Solapamiento del test con entrenamiento/validación (solo conteos de entradas; **no se usan etiquetas de test**).

In [3]:
train_raw = pd.read_csv(RAW/TRAIN_FILE)
test_raw  = pd.read_csv(RAW/TEST_FILE)
assert train_raw["id"].is_unique, "id debería identificar filas"
feats_all = feature_columns(train_raw)
assert list(train_raw.columns) == list(test_raw.columns), "Esquemas distintos entre training y testing"

def audit_file(name, df):
    p = RAW/name; sha = sha256_file(p)
    miss = df.isna().sum(); miss = miss[miss > 0]
    return {"ruta": f"data/raw/{name}", "fuente": SOURCE_INFO,
            "fecha_descarga": DOWNLOAD_DATE or dt.date.fromtimestamp(p.stat().st_mtime).isoformat(),
            "sha256": sha, "sha256_esperado_anexo": EXPECTED_SHA[name], "coincide_con_anexo": sha == EXPECTED_SHA[name],
            "filas": int(len(df)), "filas_esperadas": EXPECTED_ROWS[name], "filas_coinciden": len(df) == EXPECTED_ROWS[name],
            "columnas": int(df.shape[1]), "clases_label": {str(k): int(v) for k, v in df[TARGET].value_counts().sort_index().items()},
            "pct_ataque": round(float(df[TARGET].mean()), 4),
            "faltantes_total": int(df.isna().sum().sum()), "columnas_con_faltantes": {k: int(v) for k, v in miss.items()},
            "tipos": {k: int(v) for k, v in df.dtypes.astype(str).value_counts().items()}}
fuentes = {TRAIN_FILE: audit_file(TRAIN_FILE, train_raw), TEST_FILE: audit_file(TEST_FILE, test_raw)}
display(pd.DataFrame(fuentes).T[["filas", "filas_coinciden", "columnas", "clases_label", "pct_ataque", "faltantes_total", "coincide_con_anexo"]])

# --- Duplicados y contradicciones (por variables predictoras) -------------------------------------------
def dup_report(df):
    g = row_group_hash(df); sizes = g.value_counts()
    nun = df.groupby(g)[TARGET].nunique()
    contra = nun[nun > 1].index
    return g, {"filas_duplicadas_extra": int(g.duplicated().sum()), "grupos_con_duplicados": int((sizes > 1).sum()),
               "filas_en_grupos_duplicados": int(g.map(sizes).gt(1).sum()), "grupo_mas_grande": int(sizes.max()),
               "grupos_contradictorios": int(len(contra)), "filas_en_grupos_contradictorios": int(g.isin(contra).sum())}
g_tr, dup_tr = dup_report(train_raw); g_te, dup_te = dup_report(test_raw)
print("Duplicados por entradas — TRAIN:", dup_tr); print("Duplicados por entradas — TEST :", dup_te)
print("Variables usadas:", len(feats_all), "| excluidas:", EXCLUDE)

,filas,filas_coinciden,columnas,clases_label,pct_ataque,faltantes_total,coincide_con_anexo
C:\Users\Usuario\Documents\taller_defiende_mlops\data\UNSW_NB15_training-set.csv,175341,True,45,"{'0': 56000, '1': 119341}",0.6806,0,True
C:\Users\Usuario\Documents\taller_defiende_mlops\data\UNSW_NB15_testing-set.csv,82332,True,45,"{'0': 37000, '1': 45332}",0.5506,0,True


Duplicados por entradas — TRAIN: {'filas_duplicadas_extra': 74301, 'grupos_con_duplicados': 9748, 'filas_en_grupos_duplicados': 84049, 'grupo_mas_grande': 415, 'grupos_contradictorios': 229, 'filas_en_grupos_contradictorios': 940}
Duplicados por entradas — TEST : {'filas_duplicadas_extra': 28386, 'grupos_con_duplicados': 6318, 'filas_en_grupos_duplicados': 34704, 'grupo_mas_grande': 88, 'grupos_contradictorios': 6, 'filas_en_grupos_contradictorios': 14}
Variables usadas: 42 | excluidas: ['label', 'attack_cat', 'id']


### 3.1 Gráficas de exploración
Solo se usa el archivo de **entrenamiento** para explorar relaciones con la etiqueta.

In [4]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.3))
for a, (n, df) in zip(ax[:2], (("training-set", train_raw), ("testing-set", test_raw))):
    c = df[TARGET].value_counts().sort_index()
    bars = a.bar(["normal (0)", "ataque (1)"], c.values, color=["#2a9d8f", "#e76f51"])
    for b, v in zip(bars, c.values): a.text(b.get_x() + b.get_width()/2, v, f"{v:,}\n({v/c.sum():.1%})", ha="center", va="bottom", fontsize=9)
    a.set_title(f"Clases en {n}"); a.set_ylim(0, c.max()*1.25)
cat = train_raw["attack_cat"].value_counts()
ax[2].barh(cat.index[::-1], cat.values[::-1], color="#457b9d"); ax[2].set_title("Categorías de ataque (solo exploración; NO es entrada)")
savefig("01_clases.png")

num_cols = [c for c in feats_all if pd.api.types.is_numeric_dtype(train_raw[c])]
top = train_raw[num_cols].corrwith(train_raw[TARGET]).abs().sort_values(ascending=False).head(8).index
fig, ax = plt.subplots(2, 4, figsize=(17, 6.5))
for a, c in zip(ax.ravel(), top):
    for lab, col, nm in ((0, "#2a9d8f", "normal"), (1, "#e76f51", "ataque")):
        a.hist(np.log1p(train_raw.loc[train_raw[TARGET] == lab, c].clip(lower=0)), bins=40, alpha=.6, color=col, label=nm, density=True)
    a.set_title(f"log1p({c})")
ax[0, 0].legend(); fig.suptitle("Las 8 variables más correlacionadas con la etiqueta (train)", y=1.02)
savefig("02_distribuciones_top8.png")

corr = train_raw[list(top) + [c for c in num_cols if c not in top][:12]].corr()
fig, a = plt.subplots(figsize=(8.5, 7)); im = a.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
a.set_xticks(range(len(corr))); a.set_xticklabels(corr.columns, rotation=90, fontsize=8); a.set_yticks(range(len(corr))); a.set_yticklabels(corr.columns, fontsize=8)
plt.colorbar(im, ax=a, shrink=.8); a.set_title("Correlación entre variables numéricas seleccionadas"); a.grid(False)
savefig("03_correlacion.png")

cats = [c for c in feats_all if not pd.api.types.is_numeric_dtype(train_raw[c])]
print("Variables categóricas:", cats, "| cardinalidades:", {c: int(train_raw[c].nunique()) for c in cats})

Variables categóricas: ['proto', 'service', 'state'] | cardinalidades: {'proto': 133, 'service': 13, 'state': 9}


### 3.2 Particiones entrenamiento / validación (por grupos, con semilla)
`StratifiedGroupKFold` conserva la proporción de clases y **nunca separa filas con entradas idénticas**. Se usa un pliegue (~20 %) como validación.
Las particiones quedan **fijas** para los cuatro experimentos.

In [5]:
groups = pd.factorize(g_tr)[0]
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
tr_idx, va_idx = next(sgkf.split(train_raw, train_raw[TARGET], groups))
assert set(groups[tr_idx]).isdisjoint(set(groups[va_idx])), "¡Un grupo de entradas idénticas quedó repartido!"
train_p, val_p = train_raw.iloc[tr_idx].reset_index(drop=True), train_raw.iloc[va_idx].reset_index(drop=True)
train_p.to_csv(PREP/"train.csv", index=False); val_p.to_csv(PREP/"val.csv", index=False)
(PREP/"split_ids.json").write_text(json.dumps({"seed": SEED, "n_folds": N_FOLDS, "metodo": "StratifiedGroupKFold (grupos = hash de variables predictoras), pliegue 0 = validacion",
                                               "train_ids": train_p["id"].tolist(), "val_ids": val_p["id"].tolist()}))
splits = {"semilla": SEED, "metodo": "StratifiedGroupKFold(n_splits=5, shuffle=True), grupos = hash de variables predictoras, primer pliegue = validacion",
          "filas_train": len(train_p), "filas_val": len(val_p), "pct_ataque_train": round(float(train_p[TARGET].mean()), 4), "pct_ataque_val": round(float(val_p[TARGET].mean()), 4),
          "grupos_compartidos_train_val": 0, "sha256_ids_train": sha256_array(train_p["id"].values), "sha256_ids_val": sha256_array(val_p["id"].values),
          "sha256_train_csv": sha256_file(PREP/"train.csv"), "sha256_val_csv": sha256_file(PREP/"val.csv")}

# --- Cruces del test con train/val (SOLO entradas; no se usan etiquetas de test) -------------------------
set_tr, set_va = set(g_tr.iloc[tr_idx]), set(g_tr.iloc[va_idx])
in_tr, in_va = g_te.isin(set_tr), g_te.isin(set_va)
cruces = {"filas_test": int(len(g_te)), "filas_test_con_entrada_en_train": int(in_tr.sum()), "filas_test_con_entrada_en_val": int(in_va.sum()),
          "filas_test_con_entrada_en_train_o_val": int((in_tr | in_va).sum()), "pct_test_solapado": round(float((in_tr | in_va).mean()), 4),
          "grupos_train_val_compartidos": 0,
          "nota": "El test comparte entradas con train/val; el desempeño en test puede ser optimista. No se usaron etiquetas de test."}
print(json.dumps({"splits": splits, "cruces": cruces}, indent=2))

politica = ("No se eliminan filas (los datos se conservan tal como se publicaron). Los duplicados por entradas se agrupan y NUNCA se reparten entre train y val. "
            "Los grupos con etiquetas contradictorias se conservan y se reportan; su presencia pone un techo a la métrica alcanzable. "
            "Imputación (mediana / moda), escalado y codificación one-hot se ajustan solo con entrenamiento dentro de un Pipeline.")
auditoria = {"fuentes": fuentes, "duplicados_train": dup_tr, "duplicados_test": dup_te, "politica_limpieza": politica,
             "variables_usadas": feats_all, "variables_excluidas": EXCLUDE, "particiones": splits, "cruces_entre_particiones": cruces,
             "esquema_train_igual_test": True}
(EV/"auditoria.json").write_text(json.dumps(auditoria, indent=2, ensure_ascii=False), encoding="utf-8")
print("\n✔ evidencias/auditoria.json escrito")

{
  "splits": {
    "semilla": 42,
    "metodo": "StratifiedGroupKFold(n_splits=5, shuffle=True), grupos = hash de variables predictoras, primer pliegue = validacion",
    "filas_train": 140272,
    "filas_val": 35069,
    "pct_ataque_train": 0.6806,
    "pct_ataque_val": 0.6806,
    "grupos_compartidos_train_val": 0,
    "sha256_ids_train": "543906211dd9396769d7859d16c438ee8fbb256a69d8d342e4197419b300183f",
    "sha256_ids_val": "02caa3ae287802d534aa3b7382c29d6d7f1374a108c2699ad8806558962166c2",
    "sha256_train_csv": "870177afa2462803dfc13130dc91c035a4aa7e0d1ae2710eb975d53ac369e7f1",
    "sha256_val_csv": "06eff4012517aba974dbc5d004c057f456a1bf92ed41de9503aecf586cdbbf65"
  },
  "cruces": {
    "filas_test": 82332,
    "filas_test_con_entrada_en_train": 6514,
    "filas_test_con_entrada_en_val": 2027,
    "filas_test_con_entrada_en_train_o_val": 8541,
    "pct_test_solapado": 0.1037,
    "grupos_train_val_compartidos": 0,
    "nota": "El test comparte entradas con train/val; el desem

In [10]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
ax[0].bar(["train", "val"], [len(train_p), len(val_p)], color=["#264653", "#e9c46a"])
for i, v in enumerate([len(train_p), len(val_p)]): ax[0].text(i, v, f"{v:,}", ha="center", va="bottom")
ax[0].set_title("Tamaño de las particiones")
w = .35; x = np.arange(2)
ax[1].bar(x - w/2, [(train_p[TARGET] == 0).mean(), (val_p[TARGET] == 0).mean()], w, label="normal", color="#2a9d8f")
ax[1].bar(x + w/2, [(train_p[TARGET] == 1).mean(), (val_p[TARGET] == 1).mean()], w, label="ataque", color="#e76f51")
ax[1].set_xticks(x); ax[1].set_xticklabels(["train", "val"]); ax[1].legend(); ax[1].set_title("Proporción de clases (estratificada)")
ax[2].bar(["con entrada en\ntrain o val", "sin cruce"], [cruces["filas_test_con_entrada_en_train_o_val"], cruces["filas_test"] - cruces["filas_test_con_entrada_en_train_o_val"]], color=["#e76f51", "#2a9d8f"])
ax[2].set_title(f"Test: filas que repiten entradas de train/val ({cruces['pct_test_solapado']:.1%})")
savefig("04_particiones.png")

## 4. Segunda versión de datos: preparados y particiones (DVC)
Versiona `data/prepared` (train.csv, val.csv, split_ids.json). Si ya configuraste un **remote DVC accesible al docente**, también hace `dvc push` de ambas versiones.
> Un remote que solo existe en tu computador **no sirve** para la revisión. Ejemplos: `dvc remote add -d storage s3://bucket/ruta`, un remote de DagsHub, Google Drive o SSH (consulta la documentación de DVC para la autenticación de cada uno).

In [ ]:
sh("dvc add data/prepared")
puede_subir = False
if DVC_REMOTE_URL:
    sh(f"dvc remote add -d -f storage {DVC_REMOTE_URL}")
    # Credenciales DagsHub: se guardan SOLO en .dvc/config.local (DVC lo excluye de Git). El token no se imprime ni se escribe en el cuaderno.
    from getpass import getpass
    DAGSHUB_USER = os.environ.get("DAGSHUB_USER", "davidprogramacion04-arch")
    _tok = os.environ.get("DAGSHUB_TOKEN") or getpass("Token de DagsHub (no se guarda en el cuaderno): ")
    if _tok:
        for _k, _v in (("auth", "basic"), ("user", DAGSHUB_USER), ("password", _tok)):
            _r = subprocess.run(["dvc", "remote", "modify", "storage", "--local", _k, _v], cwd=ROOT, capture_output=True, text=True)
            if _r.returncode != 0: print("✘ Error guardando", _k, "->", _r.stderr.strip()); break
        else:
            puede_subir = True; print("✔ Credenciales de DagsHub guardadas en .dvc/config.local")
        del _tok
    else:
        print("⚠ Sin token: no se hará dvc push (las bases de datos NO quedarán en DagsHub).")
    sh("git check-ignore -v .dvc/config.local")     # debe mostrar que Git lo ignora
else:
    print("⚠ DVC_REMOTE_URL vacío: configura un remote compartido antes de entregar (necesario para la recuperación).")
sh("git add -A"); sh('git commit -m "Datos preparados y particiones versionados con DVC (version prepared)"'); sh("git tag -f data-prep-v1")
if puede_subir: sh("dvc push")
print("Referencias DVC:", dvc_reference(ROOT)); print("Versión de datos usada en MLflow:", data_version_string(ROOT))

## 5. Experimento E1 (referencia) y versión **v1**
**Hipótesis E1:** un modelo lineal con variables escaladas separa razonablemente normal/ataque (línea base).
Cada ejecución registra en MLflow: parámetros, semilla, versión de datos, commit de código, métricas de validación y el **modelo completo** (preprocesamiento + estimador).

**Métricas** (ataque = clase positiva): recall, precision, FPR (= FP/(FP+TN)), macro-F1 y matriz de confusión; accuracy solo como complemento.
**Denominador cero → 0.0** (p. ej. precision si el modelo no predice ningún ataque).

In [12]:
conf = load_configs(ROOT); CFG = {e["id"]: e for e in conf["experiments"]}
if git_dirty(ROOT): print("⚠ Hay cambios sin commit: el 'commit de código' registrado en MLflow no los incluiría. Haz commit y repite si es importante.")
train_p, val_p = load_prepared(ROOT); feats = feature_columns(train_p)
print(f"train={len(train_p):,}  val={len(val_p):,}  variables de entrada={len(feats)}")
RES = {}
RES["E1"] = run_experiment(CFG["E1"], ROOT, seed=SEED, max_fpr=MAX_FPR)
display(pd.Series(RES["E1"]["metrics_threshold_0.5"]).to_frame("E1 @ umbral 0.5").T)

train=140,272  val=35,069  variables de entrada=42


2026/10/28 15:41:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/28 15:41:20 INFO mlflow.store.db.utils: Updating database tables


[E1] run_id=666c57c9dc2d402d84398e93b72f2e4e recall=0.9878 fpr=0.1720 macroF1=0.9241 | umbral_val=0.7771 recall=0.8948 fpr=0.0500


,tn,fp,fn,tp,recall,precision,fpr,f1_attack,macro_f1,accuracy
E1 @ umbral 0.5,9274.0,1926.0,292.0,23577.0,0.987767,0.924479,0.171964,0.955076,0.924133,0.936753


### 5.1 Construir y versionar **v1**
v1 = *primera solución funcional*: preprocesamiento + regresión logística + umbral 0.5 + métricas de validación. Se empaqueta en `hf_v1/`
(modelo completo, `config.json` con umbral, `requirements.txt`, model card) y se marca en Git con el tag `v1`.

In [13]:
HF_SHAS_FILE = EV/"hf_revisiones.json"
r1, c1 = RES["E1"], CFG["E1"]
cfg_v1 = build_bundle(ROOT/"hf_v1", "v1", r1, c1, 0.5, r1["metrics_threshold_0.5"], feats, HF_REPO_ID, SEED,
    change_text="Primera solución funcional: regresión logística (C=1) con imputación + escalado + one-hot, umbral por defecto 0.5.",
    priority="un punto de partida simple y reproducible", model_src=ROOT/"models/E1/model.joblib")
print("v1 → hash del modelo:", cfg_v1["model_sha256"]); print(sorted(os.listdir(ROOT/"hf_v1")))
sh("git add -A"); sh('git commit --allow-empty -m "v1: regresion logistica base (E1), umbral 0.5"'); sh("git tag -f v1")

v1 → hash del modelo: 52fcd1592b573c074bf432be60235b1d1c05bfe34b2c50d7e3904de6464d569b
['README.md', 'config.json', 'model.joblib', 'requirements.txt']
$ git add -A

[salida: 0]

$ git commit --allow-empty -m "v1: regresion logistica base (E1), umbral 0.5"
[master 8de9978] v1: regresion logistica base (E1), umbral 0.5
[salida: 0]

$ git tag -f v1

[salida: 0]



CompletedProcess(args='git tag -f v1', returncode=0, stdout='', stderr='')

## 6. Experimentos E2–E4 y comparación justa
Las **mismas particiones** train/val para los cuatro experimentos; cada uno cambia **una cosa** respecto a su referencia (ver `configs/experimentos.yaml`).

| ID | Familia | Cambio | Hipótesis (resumen) |
|---|---|---|---|
| E1 | Regresión logística | línea base | modelo lineal razonable, FPR alta por desbalance |
| E2 | Regresión logística | `class_weight=balanced` | baja la FPR a costa de algo de recall |
| E3 | Árboles | árbol de decisión (prof. 10) | captura no linealidades |
| E4 | Árboles | bosque aleatorio (100 árboles) | reduce varianza del árbol |

In [14]:
for cid in ("E2", "E3", "E4"):
    RES[cid] = run_experiment(CFG[cid], ROOT, seed=SEED, max_fpr=MAX_FPR)

[E2] run_id=1f3a409ca3944bf4a814887f275176a1 recall=0.9522 fpr=0.1062 macroF1=0.9235 | umbral_val=0.6244 recall=0.8970 fpr=0.0500
[E3] run_id=5cb3be5734da4545816c0cbf91de8c8e recall=0.9747 fpr=0.1169 macroF1=0.9362 | umbral_val=0.6579 recall=0.9151 fpr=0.0450
[E4] run_id=e7e5fc82fafa4064806675924aceea5b recall=0.9489 fpr=0.0548 macroF1=0.9407 | umbral_val=0.5339 recall=0.9429 fpr=0.0500


In [15]:
def conclusion(cid, m, base):
    if cid == "E1": return "Referencia."
    return (f"Frente a E1 (umbral 0.5): recall {m['recall']-base['recall']:+.3f}, precision {m['precision']-base['precision']:+.3f}, "
            f"FPR {m['fpr']-base['fpr']:+.3f}, macro-F1 {m['macro_f1']-base['macro_f1']:+.3f}.")
rows = []
for cid, r in RES.items():
    m, t = r["metrics_threshold_0.5"], r["metrics_tuned"]
    rows.append({"ID": cid, "hipotesis": " ".join(r["hypothesis"].split()), "familia": r["family"], "modelo": r["model"],
                 "parametros": json.dumps(r["params"]), "semilla": r["seed"], "commit_git": r["git_commit"], "referencia_dvc": r["data_version"],
                 "run_id": r["run_id"], "umbral": 0.5, "recall": m["recall"], "precision": m["precision"], "fpr": m["fpr"], "macro_f1": m["macro_f1"],
                 "accuracy": m["accuracy"], "umbral_validacion": r["tuned_threshold"], "recall_umbral_val": t["recall"], "precision_umbral_val": t["precision"],
                 "fpr_umbral_val": t["fpr"], "macro_f1_umbral_val": t["macro_f1"], "model_sha256": r["model_sha256"],
                 "segundos_entrenamiento": round(r["fit_seconds"], 1), "conclusion_validacion": conclusion(cid, m, RES["E1"]["metrics_threshold_0.5"])})
exp_df = pd.DataFrame(rows); exp_df.to_csv(EV/"experimentos.csv", index=False, encoding="utf-8")
show = exp_df.set_index("ID")[["familia", "modelo", "recall", "precision", "fpr", "macro_f1", "accuracy", "umbral_validacion", "recall_umbral_val", "fpr_umbral_val", "macro_f1_umbral_val"]]
display(show.style.format({c: "{:.4f}" for c in show.columns if show[c].dtype == float}).background_gradient(subset=["recall", "macro_f1"], cmap="Greens").background_gradient(subset=["fpr"], cmap="Reds"))
print("✔ evidencias/experimentos.csv escrito")

,familia,modelo,recall,precision,fpr,macro_f1,accuracy,umbral_validacion,recall_umbral_val,fpr_umbral_val,macro_f1_umbral_val
ID,,,,,,,,,,,
E1,regresion_logistica,LogisticRegression,0.9878,0.9245,0.1720,0.9241,0.9368,0.7771,0.8948,0.0500,0.9034
E2,regresion_logistica,LogisticRegression,0.9522,0.9503,0.1062,0.9235,0.9336,0.6244,0.8970,0.0500,0.9050
E3,arboles,DecisionTreeClassifier,0.9747,0.9467,0.1169,0.9362,0.9455,0.6579,0.9151,0.0450,0.9197
E4,arboles,RandomForestClassifier,0.9489,0.9736,0.0548,0.9407,0.9477,0.5339,0.9429,0.0500,0.9381


✔ evidencias/experimentos.csv escrito


### 6.1 Gráficas de comparación
Izquierda: métricas con el umbral por defecto (0.5). Derecha: cada modelo con **su** umbral elegido en validación (FPR ≤ 5 %).

In [16]:
fig, ax = plt.subplots(1, 2, figsize=(16, 4.6))
exp_df.set_index("ID")[["recall", "precision", "fpr", "macro_f1"]].plot.bar(ax=ax[0], rot=0, color=["#2a9d8f", "#e9c46a", "#e76f51", "#264653"])
ax[0].set_title("Validación · umbral 0.5"); ax[0].set_ylim(0, 1.05)
t = exp_df.set_index("ID")[["recall_umbral_val", "precision_umbral_val", "fpr_umbral_val", "macro_f1_umbral_val"]]; t.columns = ["recall", "precision", "fpr", "macro_f1"]
t.plot.bar(ax=ax[1], rot=0, color=["#2a9d8f", "#e9c46a", "#e76f51", "#264653"]); ax[1].set_title(f"Validación · umbral elegido (FPR ≤ {MAX_FPR:.0%})"); ax[1].set_ylim(0, 1.05)
savefig("05_comparacion_modelos.png")

VS = {cid: pd.read_csv(ROOT/f"models/{cid}/val_scores.csv") for cid in RES}
colors = dict(zip(RES, ["#264653", "#2a9d8f", "#e9c46a", "#e76f51"]))
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
for cid, d in VS.items():
    f, tpr, _ = roc_curve(d.y_true, d.score_attack); ax[0].plot(f, tpr, color=colors[cid], label=cid); ax[1].plot(f, tpr, color=colors[cid], label=cid)
    p, r, _ = precision_recall_curve(d.y_true, d.score_attack); ax[2].plot(r, p, color=colors[cid], label=cid)
    ax[1].scatter(RES[cid]["metrics_tuned"]["fpr"], RES[cid]["metrics_tuned"]["recall"], color=colors[cid], marker="*", s=160, zorder=5)
ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[0].set_title("ROC (validación)"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("Recall de ataque")
ax[1].axvline(MAX_FPR, color="k", ls=":"); ax[1].set_xlim(0, .2); ax[1].set_title(f"Zoom ROC (FPR ≤ 0.2) · ★ umbral elegido · línea = {MAX_FPR:.0%}"); ax[1].set_xlabel("FPR")
ax[2].set_title("Precision–Recall (ataque)"); ax[2].set_xlabel("Recall"); ax[2].set_ylabel("Precision"); ax[0].legend()
savefig("06_roc_pr.png")

fig, ax = plt.subplots(1, 4, figsize=(17, 3.9))
for a, (cid, d) in zip(ax, VS.items()):
    ConfusionMatrixDisplay.from_predictions(d.y_true, apply_threshold(d.score_attack, .5), display_labels=["normal", "ataque"], ax=a, colorbar=False, cmap="Blues", values_format="d")
    a.set_title(f"{cid} · umbral 0.5"); a.grid(False)
savefig("07_matrices_confusion_E1_E4.png")

In [17]:
# Importancia de variables del mejor modelo de árboles (bosque E4)
pipe_e4 = joblib.load(ROOT/"models/E4/model.joblib")
names = pipe_e4.named_steps["prep"].get_feature_names_out()
imp = pd.Series(pipe_e4.named_steps["clf"].feature_importances_, index=names).sort_values(ascending=False).head(15)
fig, a = plt.subplots(figsize=(8, 5)); a.barh([n.split("__", 1)[-1] for n in imp.index[::-1]], imp.values[::-1], color="#e76f51")
a.set_title("E4 · 15 variables más importantes (entrenamiento)"); savefig("08_importancias_E4.png")

## 7. Selección de umbral y versión **v2** (solo con validación)
**Regla de decisión (definida de antemano):** para cada modelo se elige el umbral que **maximiza el recall de ataque con FPR ≤ 5 %** en validación
(un equipo de seguridad tolera pocas falsas alarmas). Entre modelos se elige el de mayor recall bajo esa restricción; desempata macro-F1.
**El test no interviene en ninguna decisión.**

In [18]:
cand = exp_df[exp_df.fpr_umbral_val <= MAX_FPR + 1e-9].sort_values(["recall_umbral_val", "macro_f1_umbral_val"], ascending=False)
display(cand[["ID", "modelo", "umbral_validacion", "recall_umbral_val", "precision_umbral_val", "fpr_umbral_val", "macro_f1_umbral_val"]].reset_index(drop=True))
V2_ID = cand.iloc[0]["ID"]; r2, c2 = RES[V2_ID], CFG[V2_ID]
thr2, m2, m1 = r2["tuned_threshold"], r2["metrics_tuned"], r1["metrics_threshold_0.5"]
print(f"→ v2 = {V2_ID} ({c2['model']}) con umbral {thr2:.6f}")
cmp_df = pd.DataFrame({"v1 (E1, umbral 0.5)": m1, f"v2 ({V2_ID}, umbral {thr2:.4f})": m2}).T[["tn", "fp", "fn", "tp", "recall", "precision", "fpr", "macro_f1", "accuracy"]]
display(cmp_df.style.format({c: "{:.4f}" for c in ["recall", "precision", "fpr", "macro_f1", "accuracy"]}))
mejora = [k for k in ("recall", "precision", "macro_f1") if m2[k] > m1[k]] + (["fpr (menor)"] if m2["fpr"] < m1["fpr"] else [])
empeora = [k for k in ("recall", "precision", "macro_f1") if m2[k] < m1[k]] + (["fpr (mayor)"] if m2["fpr"] > m1["fpr"] else [])
print("v2 mejora:", mejora or "nada", "| v2 empeora:", empeora or "nada")

,ID,modelo,umbral_validacion,recall_umbral_val,precision_umbral_val,fpr_umbral_val,macro_f1_umbral_val
0,E4,RandomForestClassifier,0.533862,0.942939,0.975723,0.050,0.938106
1,E3,DecisionTreeClassifier,0.657895,0.915120,0.977447,0.045,0.919748
2,E2,LogisticRegression,0.624406,0.897021,0.974512,0.050,0.904977
3,E1,LogisticRegression,0.777147,0.894843,0.974451,0.050,0.903437


→ v2 = E4 (RandomForestClassifier) con umbral 0.533862


,tn,fp,fn,tp,recall,precision,fpr,macro_f1,accuracy
"v1 (E1, umbral 0.5)",9274.000000,1926.000000,292.000000,23577.000000,0.9878,0.9245,0.1720,0.9241,0.9368
"v2 (E4, umbral 0.5339)",10640.000000,560.000000,1362.000000,22507.000000,0.9429,0.9757,0.0500,0.9381,0.9452


v2 mejora: ['precision', 'macro_f1', 'fpr (menor)'] | v2 empeora: ['recall']


In [19]:
d = VS[V2_ID]; f, tpr, thr = roc_curve(d.y_true, d.score_attack, drop_intermediate=False)
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
ax[0].plot(thr, tpr, label="Recall (ataque)", color="#2a9d8f"); ax[0].plot(thr, f, label="FPR", color="#e76f51"); ax[0].axvline(thr2, color="k", ls="--", label=f"umbral v2 = {thr2:.3f}")
ax[0].axvline(.5, color="gray", ls=":", label="umbral v1 = 0.5"); ax[0].axhline(MAX_FPR, color="#e76f51", ls=":", lw=.8)
ax[0].set_xlim(0, 1); ax[0].set_xlabel("umbral"); ax[0].set_title(f"{V2_ID}: recall y FPR vs umbral (validación)"); ax[0].legend()
for a, (ttl, y, s, th) in zip(ax[1:], (("v1", VS["E1"].y_true, VS["E1"].score_attack, .5), (f"v2 ({V2_ID})", d.y_true, d.score_attack, thr2))):
    ConfusionMatrixDisplay.from_predictions(y, apply_threshold(s, th), display_labels=["normal", "ataque"], ax=a, colorbar=False, cmap="Blues", values_format="d"); a.set_title(f"{ttl} · validación"); a.grid(False)
savefig("09_v1_vs_v2.png")

# Análisis de errores por categoría de ataque (validación; attack_cat se usa SOLO para análisis, no como entrada)
va = val_p[["id", "attack_cat", TARGET]].merge(VS[V2_ID][["id", "score_attack"]], on="id"); va["pred"] = apply_threshold(va.score_attack, thr2)
va["ok"] = va.pred == va[TARGET]
por_cat = va.groupby("attack_cat").agg(filas=("ok", "size"), correctas=("ok", "sum"), acierto=("ok", "mean")).sort_values("acierto")
fig, a = plt.subplots(figsize=(9, 4.8)); a.barh(por_cat.index, por_cat.acierto, color=["#2a9d8f" if i == "Normal" else "#e76f51" for i in por_cat.index])
for i, (v, n) in enumerate(zip(por_cat.acierto, por_cat.filas)): a.text(v + .005, i, f"{v:.1%} (n={int(n)})", va="center", fontsize=8)
a.set_xlim(0, 1.15); a.set_title(f"v2 · acierto por categoría (Normal = 1 − FPR; ataques = recall)"); savefig("10_error_por_categoria.png")
display(por_cat)

,filas,correctas,acierto
attack_cat,,,
Fuzzers,3728,2520,0.675966
Analysis,432,352,0.814815
Normal,11200,10640,0.950000
Exploits,6744,6683,0.990955
Shellcode,234,232,0.991453
DoS,2523,2518,0.998018
Reconnaissance,2008,2005,0.998506
Generic,7816,7813,0.999616
Backdoor,361,361,1.000000


### 7.1 Construir y versionar **v2**

In [20]:
dif_txt = (f"Revisión basada en los experimentos: modelo **{c2['model']}** (experimento {V2_ID}) y **umbral {thr2:.4f}** elegido en validación para maximizar el recall "
           f"con FPR ≤ {MAX_FPR:.0%}. Frente a v1 mejora: {', '.join(mejora) or 'nada'}; empeora: {', '.join(empeora) or 'nada'}.")
cfg_v2 = build_bundle(ROOT/"hf_v2", "v2", r2, c2, thr2, m2, feats, HF_REPO_ID, SEED, change_text=dif_txt,
                      priority=f"recall de ataque con FPR ≤ {MAX_FPR:.0%}", model_src=ROOT/f"models/{V2_ID}/model.joblib")
assert (cfg_v1["model_sha256"], cfg_v1["threshold"]) != (cfg_v2["model_sha256"], cfg_v2["threshold"]), "v2 debe diferir de v1"
print("v1:", cfg_v1["model_sha256"][:16], "umbral", cfg_v1["threshold"]); print("v2:", cfg_v2["model_sha256"][:16], "umbral", cfg_v2["threshold"])
print("Tamaño de modelo v1/v2 (MB):", round((ROOT/"hf_v1/model.joblib").stat().st_size/1e6, 1), round((ROOT/"hf_v2/model.joblib").stat().st_size/1e6, 1))
sh("git add -A"); sh(f'git commit -m "v2: {c2["model"]} ({V2_ID}) con umbral elegido en validacion"'); sh("git tag -f v2")

v1: 52fcd1592b573c07 umbral 0.5
v2: bc3ef11203251dd1 umbral 0.5338622443592118
Tamaño de modelo v1/v2 (MB): 0.0 6.5
$ git add -A

[salida: 0]

$ git commit -m "v2: RandomForestClassifier (E4) con umbral elegido en validacion"
[master c007fc0] v2: RandomForestClassifier (E4) con umbral elegido en validacion
 7 files changed, 5 insertions(+)
 create mode 100644 evidencias/experimentos.csv
 create mode 100644 evidencias/figuras/05_comparacion_modelos.png
 create mode 100644 evidencias/figuras/06_roc_pr.png
 create mode 100644 evidencias/figuras/07_matrices_confusion_E1_E4.png
 create mode 100644 evidencias/figuras/08_importancias_E4.png
 create mode 100644 evidencias/figuras/09_v1_vs_v2.png
 create mode 100644 evidencias/figuras/10_error_por_categoria.png
[salida: 0]

$ git tag -f v2

[salida: 0]



CompletedProcess(args='git tag -f v2', returncode=0, stdout='', stderr='')

## 8. Publicación en Hugging Face Hub (v1 y v2 por SHA inmutable)
Cada revisión contiene el **modelo completo**, `config.json` (umbral), `requirements.txt` y la model card. Se identifican por el **SHA completo** del commit (los tags `v1`/`v2` solo ayudan a leer).

**Antes de ejecutar:** crea un token con permiso de escritura en Hugging Face y expórtalo como variable de entorno `HF_TOKEN` (terminal: `export HF_TOKEN=...` / PowerShell: `$env:HF_TOKEN="..."`). **No lo escribas en el cuaderno ni lo subas a Git.**
Pon `PUBLISH_HF = True` en la sección 1 y `HF_REPO_ID = "tu_usuario/nombre-repo"`.

In [21]:
if PUBLISH_HF:
    from huggingface_hub import HfApi
    from getpass import getpass
    api = HfApi(token=os.environ.get("HF_TOKEN") or getpass("HF token (no se guarda): "))
    api.create_repo(HF_REPO_ID, repo_type="model", private=HF_PRIVATE, exist_ok=True)
    sha_v1 = publish_version(api, HF_REPO_ID, ROOT/"hf_v1", "v1", "v1: regresion logistica base (umbral 0.5)")
    sha_v2 = publish_version(api, HF_REPO_ID, ROOT/"hf_v2", "v2", f"v2: {c2['model']} con umbral elegido en validacion")
    HF_SHAS_FILE.write_text(json.dumps({"repo_id": HF_REPO_ID, "v1": sha_v1, "v2": sha_v2, "url": f"https://huggingface.co/{HF_REPO_ID}"}, indent=2))
if HF_SHAS_FILE.exists():
    HF = json.loads(HF_SHAS_FILE.read_text()); print(json.dumps(HF, indent=2))
else:
    HF = None; print("⚠ Aún no publicado en Hugging Face (PUBLISH_HF=False). Sigue el notebook; vuelve aquí antes de las secciones 9–11.")

⚠ Aún no publicado en Hugging Face (PUBLISH_HF=False). Sigue el notebook; vuelve aquí antes de las secciones 9–11.


## 9. Congelar la selección (antes de mirar el test)
`seleccion.json` guarda versión elegida, referencias, hash, umbral y razón. Se **commitea y etiqueta** (`seleccion-congelada`) **antes** de evaluar el test:
el commit/tag con su fecha es la evidencia de congelación. No se modifica después.

In [ ]:
sel = {"version_elegida": "v2", "exp_id": V2_ID, "run_id": r2["run_id"], "git_commit_modelo": r2["git_commit"], "referencia_dvc": r2["data_version"],
       "huggingface": ({"repo_id": HF["repo_id"], "sha_v2": HF["v2"], "sha_v1": HF["v1"]} if HF else "pendiente: publicar en HF antes de congelar"),
       "model_sha256": cfg_v2["model_sha256"], "umbral": thr2, "regla_umbral": "y_pred = 1 si score_attack >= umbral",
       "criterio": f"maximizar recall de ataque con FPR <= {MAX_FPR} en VALIDACION; desempate por macro-F1",
       "razon": dif_txt, "metricas_validacion": m2, "metricas_validacion_v1": m1,
       "congelacion": {"fecha_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), "tag_git": "seleccion-congelada",
                       "nota": "Commit/tag con fecha previo a cualquier evaluacion en test. No se modifica."}}
(EV/"seleccion.json").write_text(json.dumps(sel, indent=2, ensure_ascii=False), encoding="utf-8")
sh("git add -A"); sh('git commit -m "Seleccion congelada (modelo v2 y umbral) ANTES de evaluar test"'); sh("git tag -f seleccion-congelada")

## 10. Prueba de recuperación
**Entradas fijas:** 25 filas de validación (semilla fija). Se guardan las **salidas originales** de v1 y v2 cargando cada paquete desde disco.
La prueba real se hace en una **carpeta y entorno nuevos**, descargando por **SHA completo** (sección 10.2).

In [ ]:
REC = EV/"recuperacion"
muestra = val_p.sample(25, random_state=SEED).sort_values("id")
entradas = muestra[["id"] + feats]; entradas.to_csv(REC/"entradas_validacion.csv", index=False)
for v in ("v1", "v2"):
    pr = predict_with_bundle(ROOT/f"hf_{v}", entradas)
    pd.DataFrame({"id": entradas["id"].values, **pr.to_dict("list")}).to_csv(REC/f"salidas_originales_{v}.csv", index=False)
(REC/"hashes_entradas.json").write_text(json.dumps({"entradas_validacion_sha256": sha256_file(REC/"entradas_validacion.csv"),
    "n_filas": len(entradas), "model_sha256_v1": cfg_v1["model_sha256"], "model_sha256_v2": cfg_v2["model_sha256"]}, indent=2))
print("Entradas y salidas originales guardadas en", REC)
display(pd.read_csv(REC/"salidas_originales_v1.csv").head(5)); display(pd.read_csv(REC/"salidas_originales_v2.csv").head(5))
sh("git add -A"); sh('git commit -m "Entradas fijas y salidas originales para la prueba de recuperacion"')

### 10.1 Ensayo local (solo depuración; **no cuenta** como prueba)

In [ ]:
sh("python -m src.recuperar --local-v1 hf_v1 --local-v2 hf_v2")

### 10.2 Prueba REAL en entorno nuevo (hazla en una terminal, fuera de este cuaderno)
**Antes** sube todo: `git remote add origin https://github.com/<usuario>/<repo>.git` · `git push -u origin main --tags` · `dvc push` (y publica en HF, sección 8).
Ejecuta estos pasos en **otra carpeta** con un **entorno virtual nuevo** (sin las cachés de datos/modelos originales). Sustituye `<...>`.
Guarda la salida completa de la terminal en `evidencias/recuperacion/` (el script ya escribe `log_recuperacion.txt` y `resultado_recuperacion.json`).

```bash
# Linux / macOS
git clone https://github.com/<usuario>/<repo>.git recuperacion && cd recuperacion
git checkout <COMMIT_CODIGO_ELEGIDO>
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
dvc pull                    # recupera data/raw y data/prepared desde el almacenamiento entregado (DVC remoto)
sha256sum data/raw/*.csv    # compara con auditoria.json
python -m src.recuperar --repo-id <usuario>/<repo-hf> --v1-sha <SHA_V1_40_CARACTERES> --v2-sha <SHA_V2_40_CARACTERES>
```
```powershell
# Windows PowerShell
git clone https://github.com/<usuario>/<repo>.git recuperacion; cd recuperacion
git checkout <COMMIT_CODIGO_ELEGIDO>
python -m venv .venv; .\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
dvc pull
Get-FileHash data\raw\*.csv -Algorithm SHA256
python -m src.recuperar --repo-id <usuario>/<repo-hf> --v1-sha <SHA_V1_40_CARACTERES> --v2-sha <SHA_V2_40_CARACTERES>
```
Después copia `evidencias/recuperacion/` de la carpeta nueva a la del proyecto entregado.

## 11. Evaluación final en **test** (una sola vez)
Se evalúa **la solución congelada** (v2, su umbral) sobre **todo** el archivo de test. Seguridad: la celda se niega a ejecutarse si (a) ya existe `metricas-test.json`, (b) falta el tag `seleccion-congelada` o
(c) el hash del modelo no coincide con `seleccion.json`. Si tienes el modelo publicado, se evalúa **el artefacto descargado de Hugging Face por SHA** (coherencia modelo descrito = descargado = que predice).

In [ ]:
MET = EV/"metricas-test.json"
if MET.exists(): raise RuntimeError("El test ya fue evaluado. No se repite (evaluación única).")
sel = json.loads((EV/"seleccion.json").read_text(encoding="utf-8"))
tag_commit = sh("git rev-parse seleccion-congelada", show=False).stdout.strip()
assert tag_commit, "Falta el tag 'seleccion-congelada': haz la sección 9 ANTES de evaluar el test."
assert not sh("git status --porcelain evidencias/seleccion.json", show=False).stdout.strip(), "seleccion.json tiene cambios sin commit"

if HF:                                         # modelo descargado por SHA desde Hugging Face
    from huggingface_hub import snapshot_download
    bundle = Path(snapshot_download(repo_id=HF["repo_id"], revision=HF["v2"]))
else:
    bundle = ROOT/"hf_v2"; print("⚠ Evaluando el paquete local hf_v2 (aún no hay SHA de Hugging Face).")
assert sha256_file(bundle/"model.joblib") == sel["model_sha256"], "El modelo a evaluar NO es el congelado"

test_df = pd.read_csv(RAW/TEST_FILE)
pr = predict_with_bundle(bundle, test_df)
pred_test = pd.DataFrame({"row_id": [f"{TEST_FILE}#{i}" for i in range(len(test_df))], "y_true": test_df[TARGET].values,
                          "score_attack": pr.score_attack.values, "y_pred": apply_threshold(pr.score_attack.values, sel["umbral"]), "id": test_df["id"].values})
pred_test.to_csv(EV/"predicciones-test.csv", index=False)
mt = compute_metrics(pred_test.y_true, pred_test.y_pred)
met_test = {"version": "v2", "model_sha256": sel["model_sha256"], "hf_revision_sha": HF["v2"] if HF else None, "dataset": TEST_FILE,
            "dataset_sha256": sha256_file(RAW/TEST_FILE), "filas_evaluadas": int(len(test_df)), "umbral": sel["umbral"], "regla": sel["regla_umbral"],
            "clase_positiva": "ataque (label=1)", "denominador_cero": "0.0", "matriz_confusion": {k: mt[k] for k in ("tn", "fp", "fn", "tp")},
            "metricas": {k: mt[k] for k in ("recall", "precision", "fpr", "macro_f1", "f1_attack", "accuracy")},
            "seleccion_congelada_commit": tag_commit, "evaluado_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
            "advertencia": f"El {cruces['pct_test_solapado']:.1%} de las filas de test repite entradas de train/val; dataset historico; no demuestra eficacia en redes actuales."}
MET.write_text(json.dumps(met_test, indent=2, ensure_ascii=False), encoding="utf-8")
print("✔ predicciones-test.csv y metricas-test.json escritos"); print(json.dumps(met_test["metricas"], indent=2)); print(met_test["matriz_confusion"])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.7))
ConfusionMatrixDisplay.from_predictions(pred_test.y_true, pred_test.y_pred, display_labels=["normal", "ataque"], ax=ax[0], colorbar=False, cmap="Blues", values_format="d"); ax[0].set_title("TEST · v2 (umbral congelado)"); ax[0].grid(False)
f, tpr, _ = roc_curve(pred_test.y_true, pred_test.score_attack); ax[1].plot(f, tpr, color="#e76f51"); ax[1].plot([0, 1], [0, 1], "k--", lw=.8)
ax[1].scatter(mt["fpr"], mt["recall"], marker="*", s=200, color="k", zorder=5, label="operación v2"); ax[1].set_title("ROC · test"); ax[1].set_xlabel("FPR"); ax[1].set_ylabel("Recall"); ax[1].legend()
for lab, col, nm in ((0, "#2a9d8f", "normal"), (1, "#e76f51", "ataque")):
    ax[2].hist(pred_test.loc[pred_test.y_true == lab, "score_attack"], bins=40, alpha=.6, color=col, label=nm)
ax[2].axvline(sel["umbral"], color="k", ls="--", label=f"umbral {sel['umbral']:.3f}"); ax[2].set_yscale("log"); ax[2].set_title("Distribución de score_attack (test)"); ax[2].legend()
savefig("11_test_final.png")

vt = pd.DataFrame({"validación (v2)": {k: m2[k] for k in ("recall", "precision", "fpr", "macro_f1", "accuracy")}, "test (v2)": met_test["metricas"] | {}}).loc[["recall", "precision", "fpr", "macro_f1", "accuracy"]]
display(vt.style.format("{:.4f}")); vt.plot.bar(rot=0, figsize=(9, 3.8), color=["#264653", "#e76f51"], title="Validación vs test (v2)"); savefig("12_validacion_vs_test.png")

### 11.1 Añadir el resultado de test a la model card (sin cambiar el modelo)
Se sube **solo `README.md`** como un commit nuevo; el modelo congelado no cambia (se documenta el SHA evaluado y el hash del artefacto).

In [ ]:
extra = (f"\n## Evaluación final en test (una sola vez)\n"
         f"- Revisión evaluada (SHA): `{HF['v2'] if HF else 'pendiente'}` · hash del modelo: `{sel['model_sha256']}`\n- Umbral: `{sel['umbral']:.6f}` · filas: {met_test['filas_evaluadas']:,}\n"
         f"- Recall={mt['recall']:.4f} · Precision={mt['precision']:.4f} · FPR={mt['fpr']:.4f} · Macro-F1={mt['macro_f1']:.4f} · Accuracy={mt['accuracy']:.4f}\n"
         f"- TN={mt['tn']} FP={mt['fp']} FN={mt['fn']} TP={mt['tp']}\n- {met_test['advertencia']}\n")
card_path = ROOT/"hf_v2/README.md"; card_path.write_text(card_path.read_text(encoding="utf-8") + extra, encoding="utf-8")
if HF and PUBLISH_HF:
    ci = api.upload_file(path_or_fileobj=str(card_path), path_in_repo="README.md", repo_id=HF["repo_id"], repo_type="model",
                         commit_message="Anadir resultados de test a la model card (modelo sin cambios)")
    HF["docs_commit_v2"] = ci.oid; HF_SHAS_FILE.write_text(json.dumps(HF, indent=2)); print("Model card actualizada en commit", ci.oid)
else: print("Model card local actualizada (súbela cuando publiques).")

## 12. Exportación de MLflow e índice de entrega
Se exportan los cuatro runs (parámetros, métricas, tags y artefactos ligeros) a `evidencias/mlflow/`, legibles **sin** el servidor/base de datos de tu equipo.
El archivo `model.joblib` no se copia al ZIP (se recupera desde Hugging Face); su hash queda en cada JSON.

In [ ]:
setup_mlflow(ROOT)
runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME], output_format="pandas")
runs.to_csv(EV/"mlflow/runs_todos.csv", index=False, encoding="utf-8")
client = mlflow.MlflowClient()
for cid, r in RES.items():
    run = client.get_run(r["run_id"]); d = EV/"mlflow"/f"{cid}_{r['run_id']}"; d.mkdir(parents=True, exist_ok=True)
    (d/"run.json").write_text(json.dumps({"run_id": r["run_id"], "nombre": run.info.run_name, "estado": run.info.status, "params": run.data.params, "metrics": run.data.metrics,
                                          "tags": run.data.tags, "model_sha256": r["model_sha256"]}, indent=2, ensure_ascii=False), encoding="utf-8")
    mlflow.artifacts.download_artifacts(run_id=r["run_id"], artifact_path="evidencia", dst_path=str(d))
print(sorted(p.name for p in (EV/"mlflow").iterdir()))
cross = pd.read_csv(EV/"experimentos.csv")[["ID", "run_id"]].merge(runs[["run_id", "tags.exp_id"]], on="run_id", how="left"); display(cross)

In [ ]:
HFu = (HF or {}); url_hf = f"https://huggingface.co/{HFu.get('repo_id', HF_REPO_ID)}"
entrega = f"""# Índice de la entrega — MLOps · Defiende el servidor

- **Integrantes y grupo:** <NOMBRES COMPLETOS> · Grupo <A/B>
- **GitHub:** <URL_REPO> · **commit final:** <COMMIT_FINAL_COMPLETO (git rev-parse HEAD)>
- **Hugging Face (modelos):** {url_hf}

## Versiones
| | v1 | v2 |
|---|---|---|
| Commit de código | `{r1['git_commit']}` (tag `v1`) | `{r2['git_commit']}` (tag `v2`) |
| Referencia DVC | `{r1['data_version']}` | `{r2['data_version']}` |
| Run MLflow | `{r1['run_id']}` (E1) | `{r2['run_id']}` ({V2_ID}) |
| SHA Hugging Face | `{HFu.get('v1', 'PENDIENTE')}` | `{HFu.get('v2', 'PENDIENTE')}` |
| Hash del modelo | `{cfg_v1['model_sha256']}` | `{cfg_v2['model_sha256']}` |
| Umbral | {cfg_v1['threshold']} | {cfg_v2['threshold']:.6f} |

## Ubicación de evidencias
auditoría `evidencias/auditoria.json` · experimentos `evidencias/experimentos.csv` · selección `evidencias/seleccion.json` (tag `seleccion-congelada`) ·
test `evidencias/metricas-test.json`, `evidencias/predicciones-test.csv` · MLflow `evidencias/mlflow/` · recuperación `evidencias/recuperacion/` · figuras `evidencias/figuras/`
**Almacenamiento DVC:** <URL_REMOTE y forma de acceso para el docente>

## Comandos
Ver `README.md` (instalación, `dvc pull`, `python -m src.run_experiment --id E1`, `python -m src.recuperar ...`).
Tiempo/recursos medidos: <COMPLETA: CPU/RAM/SO y tiempos>  (entrenamiento por experimento en `experimentos.csv`, columna `segundos_entrenamiento`).

## Limitaciones
Dataset histórico; {cruces['pct_test_solapado']:.1%} del test repite entradas de train/val; sin garantía en redes actuales; modelos educativos.

## Declaración de IA, código externo y contribuciones
<COMPLETA: herramientas de IA usadas, para qué, qué verificaste; código adaptado y licencias; contribución de cada integrante>
"""
(ROOT/"entrega.md").write_text(entrega, encoding="utf-8"); print("✔ entrega.md generado (completa los campos <...>)")

## 13. Resumen de resultados
Las tablas y gráficas de arriba son las cifras del informe (deben coincidir con `experimentos.csv`, `seleccion.json` y `metricas-test.json`).

In [ ]:
print("== Validación: v1 vs v2 =="); display(cmp_df.style.format({c: "{:.4f}" for c in ["recall", "precision", "fpr", "macro_f1", "accuracy"]}))
print("== Test final (v2) =="); print(json.dumps(met_test["metricas"], indent=2)); print(met_test["matriz_confusion"]); print(met_test["advertencia"])
sh("git add -A"); sh('git commit -m "Evidencias finales: test, exportacion MLflow, figuras y entrega.md"')
print("\nSiguientes pasos manuales: (1) completar entrega.md, (2) prueba de recuperación en entorno nuevo (10.2), (3) git push + dvc push, (4) informe PDF, (5) dar acceso al docente.")